# Global D2+ Expected Matchup Collection

Collect patch-16.19 matchup pages from LoLalytics. Results are appended to a JSONL checkpoint immediately; **keep that file** when you refresh. To update old early-patch rates, set `REFRESH_SNAPSHOT` to a new `YYYY-MM-DD` label and explicitly switch `RUN_FULL_COLLECTION` on. Reuse the same label to resume an interrupted run. A later label fetches each matchup again once. Complete the refresh before running the merge notebook. If the prior checkpoint exists, a dated refresh is required before this notebook will make any new requests; old rates will not silently be reused.


In [ ]:
%pip install -q requests pandas pyarrow

In [ ]:
import importlib
import sys
from pathlib import Path

start_dir = Path.cwd().resolve()
search_dirs = [start_dir, *start_dir.parents]
PROJECT_DIR = next(
    (p for p in search_dirs if (p / 'data' / 'processed' / 'unique_matchups_16_19.csv').exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        'Could not find data/processed/unique_matchups_16_19.csv. '
        'Put both downloaded files in the repository, then reopen this notebook there.'
    )

sys.path.insert(0, str(PROJECT_DIR))
import lolalytics_matchup_scraper as matchup_scraper
matchup_scraper = importlib.reload(matchup_scraper)

PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
UNIQUE_MATCHUPS_PATH = PROCESSED_DIR / 'unique_matchups_16_19.csv'
print('Project directory:', PROJECT_DIR)
print('Unique-matchup input found:', UNIQUE_MATCHUPS_PATH.exists())

In [ ]:
TARGET_PATCH = '16.19'
MATCHUP_RANK = 'd2_plus'
MATCHUP_REGIONS = ('global',)  # Global D2+ as used in the analysis
REQUEST_DELAY_SECONDS = 1.25

# Explicitly opt in to HTTP requests. The default only exports the saved checkpoint.
RUN_FULL_COLLECTION = False
# For the next full-patch refresh, use a new date label, e.g. '2026-10-04'.
# Reuse that exact label if interrupted; None only collects new/errors, not old results.
REFRESH_SNAPSHOT = None

checkpoint_path = PROCESSED_DIR / f"expected_matchups_{TARGET_PATCH.replace('.', '_')}_checkpoint.jsonl"
if RUN_FULL_COLLECTION and checkpoint_path.exists() and REFRESH_SNAPSHOT is None:
    raise ValueError(
        'An earlier LoLalytics checkpoint exists. Set REFRESH_SNAPSHOT to one '
        'YYYY-MM-DD label for the final patch snapshot, then keep that exact '
        'label if you resume an interrupted run. Leave the checkpoint intact.'
    )

potential_requests = (
    len(matchup_scraper.load_unique_matchups(UNIQUE_MATCHUPS_PATH))
    * len(MATCHUP_REGIONS)
)
print(f'Unique matchup-region keys: {potential_requests:,}')
print('A new snapshot can make up to this many page requests; allow time for request delays and retries.')
print('HTTP collection enabled:', RUN_FULL_COLLECTION)
print('Refresh label:', REFRESH_SNAPSHOT)


## One-call preflight

When collection is enabled, this makes one Global D2+ request and checks whether the current page text can be parsed. With collection off, it makes no network requests.


In [ ]:
if RUN_FULL_COLLECTION:
    preflight_url, preflight = matchup_scraper.scrape_one_matchup(
        champion='Ahri',
        opponent_champion='Zed',
        lane='mid',
        patch=TARGET_PATCH,
        rank=MATCHUP_RANK,
        region='global',
    )
    assert preflight['status'] in {'ok', 'no_data'}
    print('Preflight passed:', preflight)
    print('URL:', preflight_url)
else:
    print('Preflight skipped; collection is disabled.')


## Resumable collection

Set `RUN_FULL_COLLECTION = True` above to begin. Each attempt is appended to `expected_matchups_16_19_checkpoint.jsonl`. New snapshot labels refresh both `ok` and `no_data` rows; reusing the label skips completed rows and retries errors. **Do not delete the checkpoint or merge while the snapshot has unfinished lookups.** With collection off, this cell exports the existing checkpoint and makes no requests.


In [ ]:
if RUN_FULL_COLLECTION:
    expected_matchups = matchup_scraper.collect_expected_matchups(
        unique_matchups_path=UNIQUE_MATCHUPS_PATH,
        processed_dir=PROCESSED_DIR,
        patch=TARGET_PATCH,
        rank=MATCHUP_RANK,
        regions=MATCHUP_REGIONS,
        delay_seconds=REQUEST_DELAY_SECONDS,
        refresh_snapshot=REFRESH_SNAPSHOT,
    )
else:
    stem = f"expected_matchups_{TARGET_PATCH.replace('.', '_')}"
    saved_rows = matchup_scraper.load_checkpoint(PROCESSED_DIR / f'{stem}_checkpoint.jsonl')
    expected_matchups = matchup_scraper.export_results(
        saved_rows.values(),
        PROCESSED_DIR / f'{stem}.csv',
        PROCESSED_DIR / f'{stem}.parquet',
    )
    print('Collection disabled: exported saved checkpoint; no requests made.')


## Coverage and outputs

In [ ]:
display(expected_matchups.groupby(['region', 'status'], dropna=False).size().rename('rows').to_frame())
successful = expected_matchups.loc[expected_matchups['status'].eq('ok')]
if not successful.empty:
    display(successful['matchup_games'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

if REFRESH_SNAPSHOT is not None:
    this_snapshot = expected_matchups.loc[
        expected_matchups['patch'].eq(TARGET_PATCH)
        & expected_matchups['rank'].eq(MATCHUP_RANK)
        & expected_matchups['region'].isin(MATCHUP_REGIONS)
        & expected_matchups['snapshot_id'].eq(REFRESH_SNAPSHOT)
        & expected_matchups['status'].isin(['ok', 'no_data'])
    ]
    unfinished = potential_requests - len(this_snapshot)
    print(f'Snapshot {REFRESH_SNAPSHOT}: {unfinished:,} of {potential_requests:,} lookups unfinished.')
    if unfinished:
        print('Resume with the same refresh label before running the merge notebook.')

print('CSV:', PROCESSED_DIR / 'expected_matchups_16_19.csv')
print('Parquet:', PROCESSED_DIR / 'expected_matchups_16_19.parquet')
print('Checkpoint:', PROCESSED_DIR / 'expected_matchups_16_19_checkpoint.jsonl')
